In [12]:
import pandas as pd
import json
import sys
import os
import plotly.express as px

# Import section that will work everywhere I want it to
if 'KAGGLE_KERNEL_RUN_TYPE' in os.environ:
    from kaggle_secrets import UserSecretsClient
    print('Seems like this notebook runs in Kaggle, we will not import anything additional')
else:
    try:
        from google.colab import userdata
        from google.colab import drive
        drive.mount('/content/drive')
        print('Seems like this notebook runs in Google Colab. If not - please check import sequence and change a code')
    except:
        from dotenv import load_dotenv
        load_dotenv()
        print('Seems like this notebook runs in local environment, loaded .env file')

Seems like this notebook runs in local environment, loaded .env file


In [13]:
# Getting file path depending on an environment
if 'KAGGLE_KERNEL_RUN_TYPE' in os.environ:
    print("This notebook is running on Kaggle. We are using file path for Kaggle")
    file_path = '/kaggle/input/datasets/nikolaybaakh/nasa-data-on-all-asteroids-neows/all_neos.csv'
elif 'google.colab' in sys.modules:
    print("Running in Google Colab")
    file_path = '/content/drive/MyDrive/temp_colab_data/all_neos.csv'
else:
    print("This notebook is running locally or on another platform. We are using file path for local work")
    file_path = '../data/nasa_data/all_neos.csv'

This notebook is running locally or on another platform. We are using file path for local work


# Data Preparation

In [14]:
all_neos_df = pd.read_csv(file_path)
all_neos_df.head()

,id,neo_reference_id,name,designation,nasa_jpl_url,absolute_magnitude_h,estimated_diameter,is_potentially_hazardous_asteroid,close_approach_data,orbital_data,is_sentry_object,page
0,2000433,2000433,433 Eros (A898 PA),433,https://ssd.jpl.nasa.gov/tools/sbdb_lookup.htm...,10.38,{'kilometers': {'estimated_diameter_min': 22.3...,False,"[{'close_approach_date': '1900-12-27', 'close_...","{'orbit_id': '659', 'orbit_determination_date'...",False,0
1,2000719,2000719,719 Albert (A911 TB),719,https://ssd.jpl.nasa.gov/tools/sbdb_lookup.htm...,15.59,{'kilometers': {'estimated_diameter_min': 2.02...,False,"[{'close_approach_date': '1909-08-21', 'close_...","{'orbit_id': '274', 'orbit_determination_date'...",False,0
2,2000887,2000887,887 Alinda (A918 AA),887,https://ssd.jpl.nasa.gov/tools/sbdb_lookup.htm...,13.81,{'kilometers': {'estimated_diameter_min': 4.59...,False,"[{'close_approach_date': '1974-01-04', 'close_...","{'orbit_id': '730', 'orbit_determination_date'...",False,0
3,2001036,2001036,1036 Ganymed (A924 UB),1036,https://ssd.jpl.nasa.gov/tools/sbdb_lookup.htm...,9.18,{'kilometers': {'estimated_diameter_min': 38.7...,False,"[{'close_approach_date': '1910-02-25', 'close_...","{'orbit_id': '1460', 'orbit_determination_date...",False,0
4,2001221,2001221,1221 Amor (1932 EA1),1221,https://ssd.jpl.nasa.gov/tools/sbdb_lookup.htm...,17.37,{'kilometers': {'estimated_diameter_min': 0.89...,False,"[{'close_approach_date': '1908-03-14', 'close_...","{'orbit_id': '143', 'orbit_determination_date'...",False,0


In [15]:
json.loads(all_neos_df.estimated_diameter[0].replace("'", '"'))['kilometers']['estimated_diameter_max']

49.8930414151

In [16]:
all_neos_df.estimated_diameter[-all_neos_df.estimated_diameter.apply(lambda x: isinstance(x,str))]

7113     NaN
7300     NaN
7312     NaN
7315     NaN
7416     NaN
7419     NaN
7420     NaN
24926    NaN
28983    NaN
29120    NaN
29121    NaN
29363    NaN
38501    NaN
Name: estimated_diameter, dtype: object

That's sad, but I don't think that this is important

In [17]:
all_neos_df.estimated_diameter.apply(lambda x: json.loads(x.replace("'", '"'))['kilometers']['estimated_diameter_max'] if isinstance(x, str) else None)

0        49.893041
1         4.529393
2        10.281109
3        86.704169
4         1.995446
           ...    
42121     0.708011
42122     0.567597
42123     0.847305
42124     0.835680
42125     0.812905
Name: estimated_diameter, Length: 42126, dtype: float64

In [18]:
# Converting estimated diameter to meters and getting it from json
all_neos_df['estimated_diameter_meters_max'] = all_neos_df.estimated_diameter.apply(lambda x: round(json.loads(x.replace("'", '"'))['kilometers']['estimated_diameter_max'] * 1000,2) if isinstance(x, str) else None)
all_neos_df['estimated_diameter_meters_min'] = all_neos_df.estimated_diameter.apply(lambda x: round(json.loads(x.replace("'", '"'))['kilometers']['estimated_diameter_min'] * 1000,2) if isinstance(x, str) else None)
all_neos_df.drop('estimated_diameter', axis = 1, inplace=True) # dropping json as we got all we needed

all_neos_df.head()

,id,neo_reference_id,name,designation,nasa_jpl_url,absolute_magnitude_h,is_potentially_hazardous_asteroid,close_approach_data,orbital_data,is_sentry_object,page,estimated_diameter_meters_max,estimated_diameter_meters_min
0,2000433,2000433,433 Eros (A898 PA),433,https://ssd.jpl.nasa.gov/tools/sbdb_lookup.htm...,10.38,False,"[{'close_approach_date': '1900-12-27', 'close_...","{'orbit_id': '659', 'orbit_determination_date'...",False,0,49893.04,22312.85
1,2000719,2000719,719 Albert (A911 TB),719,https://ssd.jpl.nasa.gov/tools/sbdb_lookup.htm...,15.59,False,"[{'close_approach_date': '1909-08-21', 'close_...","{'orbit_id': '274', 'orbit_determination_date'...",False,0,4529.39,2025.61
2,2000887,2000887,887 Alinda (A918 AA),887,https://ssd.jpl.nasa.gov/tools/sbdb_lookup.htm...,13.81,False,"[{'close_approach_date': '1974-01-04', 'close_...","{'orbit_id': '730', 'orbit_determination_date'...",False,0,10281.11,4597.85
3,2001036,2001036,1036 Ganymed (A924 UB),1036,https://ssd.jpl.nasa.gov/tools/sbdb_lookup.htm...,9.18,False,"[{'close_approach_date': '1910-02-25', 'close_...","{'orbit_id': '1460', 'orbit_determination_date...",False,0,86704.17,38775.28
4,2001221,2001221,1221 Amor (1932 EA1),1221,https://ssd.jpl.nasa.gov/tools/sbdb_lookup.htm...,17.37,False,"[{'close_approach_date': '1908-03-14', 'close_...","{'orbit_id': '143', 'orbit_determination_date'...",False,0,1995.45,892.39


In [19]:
# Creating orbital data as a separate DataFrame. We will not parse it since we don't know if we will need it
orbital_data = all_neos_df[['id', 'neo_reference_id', 'name', 'orbital_data']].copy()
orbital_data.head()

,id,neo_reference_id,name,orbital_data
0,2000433,2000433,433 Eros (A898 PA),"{'orbit_id': '659', 'orbit_determination_date'..."
1,2000719,2000719,719 Albert (A911 TB),"{'orbit_id': '274', 'orbit_determination_date'..."
2,2000887,2000887,887 Alinda (A918 AA),"{'orbit_id': '730', 'orbit_determination_date'..."
3,2001036,2001036,1036 Ganymed (A924 UB),"{'orbit_id': '1460', 'orbit_determination_date..."
4,2001221,2001221,1221 Amor (1932 EA1),"{'orbit_id': '143', 'orbit_determination_date'..."


In [22]:
all_neos_df['close_approach_data_parsed'] = (
    all_neos_df['close_approach_data']
    .str.replace("'", '"', regex=False)
    .apply(json.loads)
)

exploded_df = all_neos_df.explode('close_approach_data_parsed')

cad_normalized = pd.json_normalize(exploded_df['close_approach_data_parsed'])
cad_normalized

cad_normalized.index = exploded_df.index

cad_normalized['relative_velocity_kph'] = pd.to_numeric(
    cad_normalized['relative_velocity.kilometers_per_hour'])
cad_normalized['miss_distance_meters'] = pd.to_numeric(
    cad_normalized['miss_distance.kilometers']) * 1000

child_cols = ['close_approach_date','close_approach_date_full',
            'orbiting_body',
            'relative_velocity_kph', 'miss_distance_meters']
final_child_data = cad_normalized[child_cols]

parent_cols = [
    'neo_reference_id', 'id', 'name', 'absolute_magnitude_h',
    'is_sentry_object', 'estimated_diameter_meters_max',
    'is_potentially_hazardous_asteroid'
]

close_approach_data = final_child_data.join(all_neos_df[parent_cols])

# Reset index for a clean final dataframe
close_approach_data.reset_index(drop=True, inplace=True)
close_approach_data.head()

,close_approach_date,close_approach_date_full,orbiting_body,relative_velocity_kph,miss_distance_meters,neo_reference_id,id,name,absolute_magnitude_h,is_sentry_object,estimated_diameter_meters_max,is_potentially_hazardous_asteroid
0,1900-12-27,1900-Dec-27 01:30,Earth,20083.029075,4.711273e+10,2000433,2000433,433 Eros (A898 PA),10.38,False,49893.04,False
1,1907-11-05,1907-Nov-05 03:31,Earth,15820.167199,7.053323e+10,2000433,2000433,433 Eros (A898 PA),10.38,False,49893.04,False
2,1917-04-20,1917-Apr-20 21:19,Earth,17340.422466,7.468781e+10,2000433,2000433,433 Eros (A898 PA),10.38,False,49893.04,False
3,1924-03-05,1924-Mar-05 22:13,Earth,16545.797588,5.382329e+10,2000433,2000433,433 Eros (A898 PA),10.38,False,49893.04,False
4,1931-01-30,1931-Jan-30 04:07,Earth,21314.946723,2.604097e+10,2000433,2000433,433 Eros (A898 PA),10.38,False,49893.04,False


# Research

What interesting could I find on NASA's Data?

We will distribute our analysis on datasets. First dataset is asteroids themselves:
1. How much asteroids are we observe?
2. Median Magnitude of asteroids.
3. How big are asteroids on average compared to real stuff?
4. What is the biggest asteroid? How big is it?
5. How many asteroids potentially hazardous? What's the difference between hazardous and non-hazardous?
6. How much Sentry asteroids are there? And what the hell are they?

Next we will go through close_approach_data:
0. How much asteroids orbit non earth flying around us?
1. When amount of asteroids close to earth will or were the highest?
2. When amount of hazardous asteroids close to earth will or were maximum?
3. When will we see the scariest asteroid? When have we seen it?
4. What's the closest asteroids? When were they closest?
5. When did we see fastest asteroids?

I will not research orbits, it will need too much knowledge

In [23]:
# 1. How much asteroids are we observe?

all_neos_df['neo_reference_id'].nunique()

42106

In [24]:
# 2. Median Magnitude of asteroids. And stats basically

all_neos_df['absolute_magnitude_h'].describe()

count    42113.000000
mean        23.904761
std          5.654479
min          8.270000
25%         21.460000
50%         23.940000
75%         25.730000
max         99.990000
Name: absolute_magnitude_h, dtype: float64

In [25]:
# Distribution of asteroids
fig = px.histogram(
    all_neos_df, 
    x='absolute_magnitude_h',
    nbins=50,  # Adjust bin size as needed
    title='Distribution of Absolute Magnitude (H)',
    labels={'absolute_magnitude_h': 'Absolute Magnitude (H)', 'count': 'Number of NEOs'},
    marginal='box',  # Adds a box plot above the histogram for better insight
    opacity=0.8,
    template='plotly_dark'  # Use a dark theme (optional, remove for white background)
)

# Update layout for cleaner look
fig.update_layout(
    xaxis_title='Absolute Magnitude (H)',
    yaxis_title='Count',
    bargap=0.1
)

fig.show()

In [26]:
# 3. How big are asteroids on average compared to real stuff?

all_neos_df[['estimated_diameter_meters_max','estimated_diameter_meters_min']].describe()

,estimated_diameter_meters_max,estimated_diameter_meters_min
count,42113.000000,42113.000000
mean,293.796188,131.389510
std,991.400952,443.367963
min,0.000000,0.000000
25%,42.470000,18.990000
50%,96.840000,43.310000
75%,303.420000,135.690000
max,131837.810000,58959.660000


In [27]:
# Distribution of asteroids size
fig = px.histogram(
    all_neos_df, 
    x='estimated_diameter_meters_max',
    nbins=50,  # Adjust bin size as needed
    title='Distribution of Diameter in meters',
    labels={'estimated_diameter_meters_max': 'Diameter meters', 'count': 'Number of NEOs'},
    marginal='box',  # Adds a box plot above the histogram for better insight
    opacity=0.8,
    template='plotly_dark'  # Use a dark theme (optional, remove for white background)
)

# Update layout for cleaner look estimated_diameter_meters_max	estimated_diameter_meters_min
fig.update_layout(
    xaxis_title='Distribution of Diameter in meters',
    yaxis_title='Count',
    bargap=0.1
)

fig.show()

In [28]:
# 4. What is the biggest asteroid? How big is it?

filtered_neos = all_neos_df.dropna(subset=['absolute_magnitude_h'])
filtered_neos = filtered_neos[filtered_neos['absolute_magnitude_h'] != 99.99]
filtered_neos.sort_values(by='absolute_magnitude_h', ascending=True).head(10)

,id,neo_reference_id,name,designation,nasa_jpl_url,absolute_magnitude_h,is_potentially_hazardous_asteroid,close_approach_data,orbital_data,is_sentry_object,page,estimated_diameter_meters_max,estimated_diameter_meters_min,close_approach_data_parsed
28991,54028221,54028221,(2004 LA33),2004 LA33,https://ssd.jpl.nasa.gov/tools/sbdb_lookup.htm...,8.27,False,[],"{'orbit_id': '3', 'orbit_determination_date': ...",False,1449,131837.81,58959.66,[]
3,2001036,2001036,1036 Ganymed (A924 UB),1036,https://ssd.jpl.nasa.gov/tools/sbdb_lookup.htm...,9.18,False,"[{'close_approach_date': '1910-02-25', 'close_...","{'orbit_id': '1460', 'orbit_determination_date...",False,0,86704.17,38775.28,"[{'close_approach_date': '1910-02-25', 'close_..."
0,2000433,2000433,433 Eros (A898 PA),433,https://ssd.jpl.nasa.gov/tools/sbdb_lookup.htm...,10.38,False,"[{'close_approach_date': '1900-12-27', 'close_...","{'orbit_id': '659', 'orbit_determination_date'...",False,0,49893.04,22312.85,"[{'close_approach_date': '1900-12-27', 'close_..."
14,2001866,2001866,1866 Sisyphus (1972 XA),1866,https://ssd.jpl.nasa.gov/tools/sbdb_lookup.htm...,12.48,False,"[{'close_approach_date': '1959-11-17', 'close_...","{'orbit_id': '1158', 'orbit_determination_date...",False,0,18968.81,8483.11,"[{'close_approach_date': '1959-11-17', 'close_..."
80,2004954,2004954,4954 Eric (1990 SQ),4954,https://ssd.jpl.nasa.gov/tools/sbdb_lookup.htm...,12.56,False,"[{'close_approach_date': '1939-10-06', 'close_...","{'orbit_id': '1011', 'orbit_determination_date...",False,4,18282.69,8176.27,"[{'close_approach_date': '1939-10-06', 'close_..."
26416,54180023,54180023,(2006 BP140),2006 BP140,https://ssd.jpl.nasa.gov/tools/sbdb_lookup.htm...,12.70,False,[],"{'orbit_id': 'E2021-PC0', 'orbit_determination...",False,1320,17141.15,7665.76,[]
8,2001627,2001627,1627 Ivar (1929 SH),1627,https://ssd.jpl.nasa.gov/tools/sbdb_lookup.htm...,12.79,False,"[{'close_approach_date': '1929-07-19', 'close_...","{'orbit_id': '1501', 'orbit_determination_date...",False,0,16445.23,7354.53,"[{'close_approach_date': '1929-07-19', 'close_..."
48,2003552,2003552,3552 Don Quixote (1983 SA),3552,https://ssd.jpl.nasa.gov/tools/sbdb_lookup.htm...,13.04,False,"[{'close_approach_date': '1920-12-29', 'close_...","{'orbit_id': '299', 'orbit_determination_date'...",False,2,14656.83,6554.73,"[{'close_approach_date': '1920-12-29', 'close_..."
31,2002212,2002212,2212 Hephaistos (1978 SB),2212,https://ssd.jpl.nasa.gov/tools/sbdb_lookup.htm...,13.49,False,"[{'close_approach_date': '1905-03-14', 'close_...","{'orbit_id': '780', 'orbit_determination_date'...",False,1,11913.52,5327.89,"[{'close_approach_date': '1905-03-14', 'close_..."
26408,54179905,54179905,(2000 QZ258),2000 QZ258,https://ssd.jpl.nasa.gov/tools/sbdb_lookup.htm...,13.61,False,[],"{'orbit_id': 'E2021-PC0', 'orbit_determination...",False,1320,11273.01,5041.44,[]


In [29]:
# 5. How many asteroids potentially hazardous? What's the difference between hazardous and non-hazardous?
all_neos_df.groupby('is_potentially_hazardous_asteroid')[['absolute_magnitude_h','estimated_diameter_meters_max']].describe()

absolute_magnitude_h                       \
                                                 count       mean       std   
is_potentially_hazardous_asteroid                                             
False                                          39389.0  24.158015  5.749754   
True                                            2724.0  20.242721  1.387449   

                                                                      \
                                     min    25%    50%    75%    max   
is_potentially_hazardous_asteroid                                      
False                               8.27  22.00  24.20  25.89  99.99   
True                               14.08  19.47  20.53  21.32  22.57   

                                  estimated_diameter_meters_max              \
                                                          count        mean   
is_potentially_hazardous_asteroid                                             
False                                                   39389.0  266.640736   
True                                                     2724.0  686.463631   

                                                                        \
                                           std     min     25%     50%   
is_potentially_hazardous_asteroid                                        
False                              1002.549558    0.00   39.45   85.91   
True                                704.821571  181.99  323.62  465.63   

                                                      
                                      75%        max  
is_potentially_hazardous_asteroid                     
False                              236.61  131837.81  
True                               758.65    9079.04

In [30]:
# 6. How much Sentry asteroids are there? And what the hell are they?
all_neos_df.groupby('is_sentry_object')[['absolute_magnitude_h','estimated_diameter_meters_max']].describe()

absolute_magnitude_h                                      \
                                count       mean       std     min    25%   
is_sentry_object                                                            
False                         40040.0  23.730277  5.729447   8.270  21.31   
True                           2073.0  27.274921  1.882923  16.767  26.20   

                                     estimated_diameter_meters_max  \
                    50%   75%    max                         count   
is_sentry_object                                                     
False             23.77  25.5  99.99                       40040.0   
True              27.38  28.5  32.56                        2073.0   

                                                                        \
                        mean          std   min    25%     50%     75%   
is_sentry_object                                                         
False             307.209216  1014.705327  0.00  47.21  104.72  325.12   
True               34.723517    96.501621  1.83  11.86   19.86   34.20   

                             
                        max  
is_sentry_object             
False             131837.81  
True                2634.15

## close to earth

In [31]:
close_approach_data.head()

,close_approach_date,close_approach_date_full,orbiting_body,relative_velocity_kph,miss_distance_meters,neo_reference_id,id,name,absolute_magnitude_h,is_sentry_object,estimated_diameter_meters_max,is_potentially_hazardous_asteroid
0,1900-12-27,1900-Dec-27 01:30,Earth,20083.029075,4.711273e+10,2000433,2000433,433 Eros (A898 PA),10.38,False,49893.04,False
1,1907-11-05,1907-Nov-05 03:31,Earth,15820.167199,7.053323e+10,2000433,2000433,433 Eros (A898 PA),10.38,False,49893.04,False
2,1917-04-20,1917-Apr-20 21:19,Earth,17340.422466,7.468781e+10,2000433,2000433,433 Eros (A898 PA),10.38,False,49893.04,False
3,1924-03-05,1924-Mar-05 22:13,Earth,16545.797588,5.382329e+10,2000433,2000433,433 Eros (A898 PA),10.38,False,49893.04,False
4,1931-01-30,1931-Jan-30 04:07,Earth,21314.946723,2.604097e+10,2000433,2000433,433 Eros (A898 PA),10.38,False,49893.04,False


In [32]:
# 0. How much asteroids orbit non earth flying around us?
close_approach_data.groupby('orbiting_body')['neo_reference_id'].nunique()

orbiting_body
Ceres        1
Earth    39053
Juptr     6202
Mars     10016
Merc      1543
Moon      5551
Satrn        5
Venus     8420
Name: neo_reference_id, dtype: int64

In [33]:
# 1. When amount of asteroids close to earth will or were the highest?

# Ensure close_approach_date is datetime
close_approach_data['close_approach_date'] = pd.to_datetime(close_approach_data['close_approach_date'])

# Group by month and count unique neo_reference_id
monthly_counts = close_approach_data.groupby(close_approach_data['close_approach_date'].dt.to_period('M'))['neo_reference_id'].nunique().reset_index(name='asteroid_count')
monthly_counts['close_approach_date'] = monthly_counts['close_approach_date'].dt.to_timestamp()

# Create the Plotly line graph
fig = px.line(monthly_counts, x='close_approach_date', y='asteroid_count', 
              title='Number of Unique Asteroids Close to Earth per Month',
              labels={'close_approach_date': 'Month', 'asteroid_count': 'Unique Asteroids Count'},
            template='plotly_dark')
fig.show()

In [43]:
# 2. When amount of hazardous asteroids close to earth will or were maximum?

hazardous_data = close_approach_data[close_approach_data['is_potentially_hazardous_asteroid'] == True].copy()
hazardous_data['year'] = hazardous_data['close_approach_date'].dt.to_period('Y').astype(str)

yearly_stats = hazardous_data.groupby('year')['neo_reference_id'].nunique().reset_index()

fig = px.line(yearly_stats, x='year', y='neo_reference_id', 
             title='Unique Potentially Hazardous Asteroids per Year',
             labels={'year': 'Year', 'neo_reference_id': 'Unique Asteroid Count'},
             template='plotly_dark')
fig.show()


In [ ]:
# 3. When will we see the scariest asteroid? When have we seen it?




In [ ]:
# 4. What's the closest asteroids? When were they closest?


In [ ]:
# 5. When did we see fastest asteroids?